# Module 4: Session Managers (10 min)

Add file-based persistence to the customer service agent. Stop the agent, restart it, and watch it remember the previous conversation.

**Prerequisites:** Modules 1-3 completed

In [1]:
!pip install -q -r requirements.txt

In [ ]:
# AWS-sponsored events / AWS credits
# If you are running this workshop with AWS-provided credits, those credits
# only work with Amazon Nova models — not Claude (the default).
# To switch, import BedrockModel and pass it to Agent(...):
#
# from strands.models import BedrockModel
# model = BedrockModel(model_id="amazon.nova-pro-v1:0")
# agent = Agent(model=model, tools=[...], session_manager=..., system_prompt=...)
#
# Available Nova model IDs: https://docs.aws.amazon.com/bedrock/latest/userguide/model-cards-amazon.html
#   amazon.nova-micro-v1:0  — fastest, text-only, lowest cost
#   amazon.nova-lite-v1:0   — low-cost, multimodal (text, image, video)
#   amazon.nova-pro-v1:0    — balanced accuracy/speed, multimodal (recommended)
#
# Run locally without AWS credentials using Ollama (https://ollama.com/download):
#   1. Install Ollama — macOS: DMG at ollama.com | Linux: curl -fsSL https://ollama.com/install.sh | sh
#   2. ollama pull llama3.1   (supports tool use)
#   3. pip install strands-agents[ollama]
#   4. from strands.models import OllamaModel
#      model = OllamaModel(host="http://localhost:11434", model_id="llama3.1")
#      agent = Agent(model=model, tools=[...], session_manager=..., system_prompt=...)
#   Other models with tool support: llama3.2, qwen2.5, qwen3, mistral

---

## Part 1: Agent Without Persistence (The Problem)

By default, agents lose their memory when you recreate them.

In [2]:
from strands import Agent, AgentSkills
from customer_service_tools import lookup_customer, get_order_history, process_refund

SYSTEM_PROMPT = """You are a customer service agent for an online electronics store.
Be helpful, professional, and concise.

If there are previous messages in the conversation history, use that context
to continue helping the customer without asking them to repeat information."""

# First interaction
agent = Agent(
    tools=[lookup_customer, get_order_history, process_refund],
    system_prompt=SYSTEM_PROMPT,
)
agent("Hi, I'm customer C-1001. Can you look up my account?")

print(f"\n📝 Messages stored: {len(agent.messages)}")

Sure! Let me pull up your account right away.
Tool #1: lookup_customer
I found your account! Here are your details:

- **Name:** Sarah Johnson
- **Email:** sarah.johnson@email.com
- **Phone:** 555-0142
- **Account Status:** Active ✅

How can I help you today, Sarah?
📝 Messages stored: 4


In [3]:
# Simulate a "restart" — create a new agent instance
agent2 = Agent(
    tools=[lookup_customer, get_order_history, process_refund],
    system_prompt=SYSTEM_PROMPT,
)

print(f"Messages after restart: {len(agent2.messages)}")
# The agent has no memory of the previous conversation!
agent2("What was my account status again?")

Messages after restart: 0
I don't have your customer information on file yet in our conversation. Could you please provide me with your **customer ID** so I can look up your account?

AgentResult(stop_reason='end_turn', message={'role': 'assistant', 'content': [{'text': "I don't have your customer information on file yet in our conversation. Could you please provide me with your **customer ID** so I can look up your account?"}], 'metadata': {'usage': {'inputTokens': 809, 'outputTokens': 36, 'totalTokens': 845}, 'metrics': {'latencyMs': 1280, 'timeToFirstByteMs': 750}}, 'tracking_id': '335e5c1e-973f-4e33-8dd8-ab37a8b60ab3'}, metrics=EventLoopMetrics(cycle_count=1, tool_metrics={}, cycle_durations=[1.2984108924865723], agent_invocations=[AgentInvocation(cycles=[EventLoopCycleMetric(event_loop_cycle_id='302971f9-ba82-40b2-b59e-57733236c547', usage={'inputTokens': 809, 'outputTokens': 36, 'totalTokens': 845})], usage={'inputTokens': 809, 'outputTokens': 36, 'totalTokens': 845})], traces=[<strands.telemetry.metrics.Trace object at 0xffff4c1bbf00>], accumulated_usage={'inputTokens': 809, 'outputTokens': 36, 'totalTokens': 845}, accumulated_metrics={'latencyMs': 1280}), st

---

## Part 2: Add FileSessionManager

The `FileSessionManager` saves conversation history to disk. On restart, it reloads the messages.

The agent below uses **two** complementary pieces:
- **Session manager** (`FileSessionManager`) - persists the conversation *outside* the process so it survives restarts. This module's focus.
- **Context manager** (`context_manager="auto"`) - controls what's kept *in context* on each call. It offloads large tool results, compresses old messages into summaries, and fires proactive compression at 85% usage.

They solve different problems and work together: one stores history, the other manages what the model sees.

> **Tip:** `context_manager="auto"` is the recommended default. For finer control you can build a custom pipeline with `ContextManager` and `Offload` strategies. See the [Context Management docs](https://strandsagents.com/docs/user-guide/concepts/context-management/).

In [4]:
from strands.session.file_session_manager import FileSessionManager

# Clean up any previous session files
import shutil, os
if os.path.exists("./sessions"):
    shutil.rmtree("./sessions")

session_manager = FileSessionManager(
    session_id="customer-session-001",
    storage_dir="./sessions",
)

agent = Agent(
    tools=[lookup_customer, get_order_history, process_refund],
    plugins=[AgentSkills(skills=["./skills"])],
    system_prompt=SYSTEM_PROMPT,
    # context_manager="auto" offloads large tool results, compresses old
    # messages into summaries, and fires proactive compression at 85% usage.
    context_manager="auto",
    session_manager=session_manager,
)

# First interaction — this gets saved to disk
agent("Hi, I'm customer C-1001. Can you look up my account?")
print(f"\n📁 Session saved. Messages: {len(agent.messages)}")

Sure! Let me look up your account right away.
Tool #1: lookup_customer
I found your account! Here's a summary:

- **Name:** Sarah Johnson
- **Email:** sarah.johnson@email.com
- **Phone:** 555-0142
- **Account Status:** Active ✅

How can I help you today, Sarah?
📁 Session saved. Messages: 4


---

## Part 3: Restart and Remember

Create a brand new agent with the same session ID. It should remember everything.

In [5]:
# Simulate restart — new agent, same session_id
session_manager_2 = FileSessionManager(
    session_id="customer-session-001",
    storage_dir="./sessions",
)

agent_restarted = Agent(
    tools=[lookup_customer, get_order_history, process_refund],
    plugins=[AgentSkills(skills=["./skills"])],
    system_prompt=SYSTEM_PROMPT,
    context_manager="auto",
    session_manager=session_manager_2,
)

print(f"🔄 Restored messages: {len(agent_restarted.messages)}")
print("The agent remembers the previous conversation!\n")

# Ask a follow-up — the agent should know we're C-1001
agent_restarted("What orders do I have? You should already know my customer ID.")

unable to find previously injected skills XML in system prompt, re-appending


🔄 Restored messages: 4
The agent remembers the previous conversation!

Of course!
Tool #1: get_order_history
Here are your orders, Sarah:

1. **ORD-5521** — Wireless Headphones — $79.99
   - Status: ✅ Delivered
   - Ordered: April 20, 2025 | Delivered: April 28, 2025
   - Tracking: TRK-998877

2. **ORD-5488** — USB-C Hub — $45.00
   - Status: 📦 Shipped
   - Ordered: May 1, 2025 | Est. Delivery: May 6, 2025
   - Tracking: TRK-887766

Is there anything you'd like to do with any of these orders?

AgentResult(stop_reason='end_turn', message={'role': 'assistant', 'content': [{'text': "Here are your orders, Sarah:\n\n1. **ORD-5521** — Wireless Headphones — $79.99\n   - Status: ✅ Delivered\n   - Ordered: April 20, 2025 | Delivered: April 28, 2025\n   - Tracking: TRK-998877\n\n2. **ORD-5488** — USB-C Hub — $45.00\n   - Status: 📦 Shipped\n   - Ordered: May 1, 2025 | Est. Delivery: May 6, 2025\n   - Tracking: TRK-887766\n\nIs there anything you'd like to do with any of these orders?"}], 'metadata': {'usage': {'inputTokens': 1901, 'outputTokens': 166, 'totalTokens': 2067}, 'metrics': {'latencyMs': 1915, 'timeToFirstByteMs': 812}}, 'tracking_id': '8bc9db81-4b72-4863-b125-89101e82a9ae'}, metrics=EventLoopMetrics(cycle_count=2, tool_metrics={'get_order_history': ToolMetrics(tool={'toolUseId': 'tooluse_tFK4TeKuKoYMw01xNAajLf', 'name': 'get_order_history', 'input': {'customer_id': 'C-1001'}}, call_count=1, success_count=1, error_count=0, total_time=0.0005884170532226562)}, cycle_durations=[

---

## 🎯 Try It Yourself

Check what's stored on disk:

In [6]:
import json

# FileSessionManager stores data in nested folders:
#   ./sessions/session_<id>/session.json
#   ./sessions/session_<id>/agents/agent_<id>/agent.json
#   ./sessions/session_<id>/agents/agent_<id>/messages/message_*.json
# So walk the tree recursively instead of listing only the top level.
session_dir = "./sessions"
for root, _dirs, files in os.walk(session_dir):
    for f in sorted(files):
        if f.endswith(".json"):
            filepath = os.path.join(root, f)
            rel = os.path.relpath(filepath, session_dir)
            print(f"📄 {rel} ({os.path.getsize(filepath)} bytes)")

# Count how many message files were persisted
message_files = [
    os.path.join(root, f)
    for root, _dirs, files in os.walk(session_dir)
    for f in files
    if f.startswith("message_") and f.endswith(".json")
]
print(f"\n💬 Messages persisted to disk: {len(message_files)}")

📄 session_customer-session-001/session.json (173 bytes)
📄 session_customer-session-001/agents/agent_default/agent.json (1286 bytes)
📄 session_customer-session-001/agents/agent_default/messages/message_0.json (360 bytes)
📄 session_customer-session-001/agents/agent_default/messages/message_1.json (793 bytes)
📄 session_customer-session-001/agents/agent_default/messages/message_2.json (595 bytes)
📄 session_customer-session-001/agents/agent_default/messages/message_3.json (735 bytes)
📄 session_customer-session-001/agents/agent_default/messages/message_4.json (370 bytes)
📄 session_customer-session-001/agents/agent_default/messages/message_5.json (759 bytes)
📄 session_customer-session-001/agents/agent_default/messages/message_6.json (735 bytes)
📄 session_customer-session-001/agents/agent_default/messages/message_7.json (950 bytes)

💬 Messages persisted to disk: 8


---

## Part 4: Snapshot Session Manager (recommended for new sessions)

`FileSessionManager` writes each message as its own record on disk. The newer `SnapshotSessionManager` instead persists the **whole agent as a single atomic blob** on each save. It's simpler, faster, and supports immutable checkpoints you can restore to.

It uses the unified `Storage` backend, so you swap `LocalFileStorage` (dev) for `S3Storage` (production) without changing your session logic. The record-based managers still work and are needed for multi-agent (Graph/Swarm) persistence.

> **When to use which:** `SnapshotSessionManager` for new single-agent sessions; `FileSessionManager`/`S3SessionManager` for existing record-based sessions or multi-agent orchestration.

In [7]:
from strands.session import SnapshotSessionManager
from strands.storage import LocalFileStorage

# Fresh directory so this snapshot demo doesn't mix with the record-based files above
if os.path.exists("./snapshot_sessions"):
    shutil.rmtree("./snapshot_sessions")

snapshot_manager = SnapshotSessionManager(
    session_id="customer-session-001",
    storage=LocalFileStorage("./snapshot_sessions"),
)

snapshot_agent = Agent(
    tools=[lookup_customer, get_order_history, process_refund],
    plugins=[AgentSkills(skills=["./skills"])],
    system_prompt=SYSTEM_PROMPT,
    context_manager="auto",
    session_manager=snapshot_manager,
)

# First interaction — persisted as a single atomic snapshot
snapshot_agent("Hi, I'm customer C-1001. Can you look up my account?")
print(f"\n📸 Snapshot saved. Messages: {len(snapshot_agent.messages)}")

# Simulate a restart — new agent, same session_id and storage
restored_agent = Agent(
    tools=[lookup_customer, get_order_history, process_refund],
    plugins=[AgentSkills(skills=["./skills"])],
    system_prompt=SYSTEM_PROMPT,
    context_manager="auto",
    session_manager=SnapshotSessionManager(
        session_id="customer-session-001",
        storage=LocalFileStorage("./snapshot_sessions"),
    ),
)
print(f"🔄 Restored from snapshot: {len(restored_agent.messages)} message(s)")

Sure! Let me look up your account right away.
Tool #1: lookup_customer
I found your account! Here's a summary:

- **Name:** Sarah Johnson
- **Email:** sarah.johnson@email.com
- **Phone:** 555-0142
- **Account Status:** Active ✅

How can I help you today, Sarah?
📸 Snapshot saved. Messages: 4
🔄 Restored from snapshot: 4 message(s)


### Session vs memory

Session managers persist the **conversation** within a single session so the agent can resume after a restart. For durable knowledge that persists **across** sessions (user preferences, facts, past decisions), Strands has a separate **memory** layer.

- **Session** = resume where you left off.
- **Memory** = durable knowledge across sessions, without replaying old conversations.

See the [Memory docs](https://strandsagents.com/docs/user-guide/concepts/memory/overview/) for more.

---

## 💬 Want a real multi-turn conversation?

In a notebook, each cell is a **single turn**. To chat back and forth with the **persistent** agent, run a companion script in a **terminal**. From the cloned repo:

```bash
cd samples/04-session-managers
pip install -r requirements.txt
python snapshot_chat.py   # recommended: SnapshotSessionManager
python chat.py            # record-based: FileSessionManager
```

Type your messages, and `quit` (or Ctrl+C) to exit. This is also the persistence demo: quit and run the same script again — it restores the earlier conversation. Use `--session-id <name>` to keep separate sessions.

---

## What's Next

The agent is persistent and follows rules — it's complete enough to ship. In **Module 5: Deploy**, you'll package this same agent and deploy it to Amazon Bedrock AgentCore Runtime with a single CLI command.